In [1]:
import pandas as pd
books = pd.read_csv("/Users/harshiniramasamy/PycharmProjects/book-recommender/books_with_categories.csv")

In [2]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k = None, device = "mps")
classifier("I love this!")

/Users/harshiniramasamy/PycharmProjects/book-recommender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 39515.65it/s]


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611993182450533},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [3]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [4]:
classifier(books["description"][0])


[[{'label': 'fear', 'score': 0.6548416018486023},
  {'label': 'neutral', 'score': 0.16985200345516205},
  {'label': 'sadness', 'score': 0.11640852689743042},
  {'label': 'surprise', 'score': 0.020700637251138687},
  {'label': 'disgust', 'score': 0.0191007312387228},
  {'label': 'joy', 'score': 0.015161268413066864},
  {'label': 'anger', 'score': 0.0039351521991193295}]]

In [5]:
sentences = [s.strip() for s in books["description"][0].split(".") if s.strip()]
predictions = classifier(sentences)

In [6]:
 sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives'

In [7]:
predictions[0]


[{'label': 'surprise', 'score': 0.7296029925346375},
 {'label': 'neutral', 'score': 0.1403854787349701},
 {'label': 'fear', 'score': 0.06816214323043823},
 {'label': 'joy', 'score': 0.04794241860508919},
 {'label': 'anger', 'score': 0.009156347252428532},
 {'label': 'disgust', 'score': 0.002628472400829196},
 {'label': 'sadness', 'score': 0.0021221612114459276}]

In [8]:
sorted (predictions[0],key = lambda x:x["label"])

[{'label': 'anger', 'score': 0.009156347252428532},
 {'label': 'disgust', 'score': 0.002628472400829196},
 {'label': 'fear', 'score': 0.06816214323043823},
 {'label': 'joy', 'score': 0.04794241860508919},
 {'label': 'neutral', 'score': 0.1403854787349701},
 {'label': 'sadness', 'score': 0.0021221612114459276},
 {'label': 'surprise', 'score': 0.7296029925346375}]

In [9]:
import numpy as np

emotions_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotions_scores = {label: [] for label in emotions_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotions_labels}
    sorted_labels = sorted(emotions_labels)
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(sorted_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [10]:
import numpy as np
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    sorted_labels = sorted(emotion_labels)
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(sorted_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = [s for s in books["description"][i].split(".") if s.strip()]
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [08:48<00:00,  9.84it/s]  


In [11]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [12]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.040478,0.273590,0.928169,0.932798,0.967158,0.729603,0.646216,9780002005883
1,0.612618,0.348285,0.942528,0.704421,0.061800,0.252545,0.887939,9780002261982
2,0.041301,0.024569,0.972321,0.767236,0.010860,0.009796,0.042176,9780006178736
3,0.351483,0.150722,0.360708,0.251881,0.049767,0.029683,0.732686,9780006280897
4,0.081412,0.184495,0.095043,0.035207,0.475881,0.074878,0.884389,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.148209,0.030643,0.919165,0.255169,0.980877,0.030656,0.853722,9788172235222
5193,0.057145,0.114383,0.022974,0.400263,0.016023,0.227765,0.883198,9788173031014
5194,0.009997,0.009929,0.339216,0.947779,0.066685,0.057625,0.375756,9788179921623
5195,0.032070,0.076714,0.459268,0.759455,0.368110,0.067808,0.951104,9788185300535


In [ ]:
books.to_csv("books_with_emotions.csv",index = False)

In [13]:
print(books.columns.tolist())

['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'title_and_subtitle', 'tagged_description', 'simple_categories']
